# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 3.1.0


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string
activity_year    string
lei              string
county_code      string
action_taken     string
loan_amount      string
income           string
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [3]:

import pandas as pd

def clean_export(source):
    # Сохраняем входные данные без изменений
    data = source.copy(deep=True)
    input_count = len(data)

    # Убираем пробелы по краям текстовых значений
    data["county_code"] = data["county_code"].astype("string").str.strip()

    # Проверяем конфликты: один source_row, но разные версии данных
    known_rows = data.dropna(subset=["source_row"])
    conflict_ids = set(
        known_rows.loc[
            known_rows.duplicated("source_row", keep=False), "source_row"
        ].unique()
    )

    conflicting = data["source_row"].isin(conflict_ids)

    # Числовые преобразования
    amount_original = data["loan_amount"].copy()
    income_original = data["income"].copy()

    data["loan_amount"] = pd.to_numeric(
        data["loan_amount"], errors="coerce"
    )
    data["income"] = pd.to_numeric(
        data["income"], errors="coerce"
    )

    amount_errors = amount_original.notna() & data["loan_amount"].isna()
    income_errors = income_original.notna() & data["income"].isna()

    # Условия качества
    row_known = data["source_row"].notna()
    action_ok = data["action_taken"].astype("string").isin(["1", "2", "3"])
    amount_ok = data["loan_amount"].notna() & data["loan_amount"].gt(0)
    county_ok = data["county_code"].eq("11001").fillna(False)
    year_ok = data["activity_year"].astype("string").eq("2023").fillna(False)

    # Конфликтующие версии и недопустимые строки отправляем в карантин
    valid = (
        row_known
        & ~conflicting
        & action_ok
        & amount_ok
        & county_ok
        & year_ok
    )

    # Убираем одинаковые повторные доставки
    candidates = data.loc[valid].copy()
    duplicate_mask = candidates.duplicated(keep="first")
    replays = int(duplicate_mask.sum())
    clean = candidates.loc[~duplicate_mask].copy()

    # Формируем карантин с исходными значениями и причинами
    quarantine = data.loc[~valid].copy()
    reasons = []

    for idx in quarantine.index:
        row_reasons = []

        if pd.isna(data.at[idx, "source_row"]):
            row_reasons.append("missing_source_row")
        elif data.at[idx, "source_row"] in conflict_ids:
            row_reasons.append("conflicting_source_row")

        if not bool(action_ok.loc[idx]):
            row_reasons.append("invalid_action_taken")
        if not bool(amount_ok.loc[idx]):
            row_reasons.append("invalid_loan_amount")
        if not bool(county_ok.loc[idx]):
            row_reasons.append("invalid_county_code")
        if not bool(year_ok.loc[idx]):
            row_reasons.append("invalid_activity_year")

        reasons.append("; ".join(row_reasons) or "quality_rule_failed")

    quarantine["reason"] = reasons

    audit = {
        "input": input_count,
        "replays": replays,
        "clean": len(clean),
        "quarantine": len(quarantine),
        "parse_errors": {
            "loan_amount": int(amount_errors.sum()),
            "income": int(income_errors.sum())
        }
    }

    return clean, quarantine, audit


## H2. Год и известный доход

In [4]:

raw_copy = export.copy(deep=True)

clean, quarantine, audit = clean_export(export)

print("Количество подготовленных записей:", len(clean))
print("Известный income:", clean["income"].notna().sum())
print("Неизвестный income:", clean["income"].isna().sum())

print(
    "Средний известный доход:",
    clean["income"].mean()
)
print("Единица измерения income: тысячи USD в год")

print("\nЖурнал качества:")
print(audit)

print(
    "Баланс выполняется:",
    audit["input"] == (
        audit["replays"]
        + audit["clean"]
        + audit["quarantine"]
    )
)

print("Входные данные не изменены:", export.equals(raw_copy))
print(
    'Во всех подготовленных строках год равен "2023":',
    clean["activity_year"].astype("string").eq("2023").all()
)


Количество подготовленных записей: 6826
Известный income: 6483
Неизвестный income: 343
Средний известный доход: 237.08514576584915
Единица измерения income: тысячи USD в год

Журнал качества:
{'input': 6832, 'replays': 0, 'clean': 6826, 'quarantine': 6, 'parse_errors': {'loan_amount': 1, 'income': 1}}
Баланс выполняется: True
Входные данные не изменены: True
Во всех подготовленных строках год равен "2023": True


## H3. Проверка отчётного года

In [5]:

wrong_year = export.copy(deep=True)

wrong_year.loc[wrong_year.index[0], "activity_year"] = "2022"

year_ok = (
    wrong_year["activity_year"]
    .astype("string")
    .eq("2023")
    .fillna(False)
    .all()
)

print("Проверка года пройдена:", year_ok)
print("Нарушение обнаружено:", not year_ok)
print("Исходный export не изменился:", export.equals(raw_copy))


Проверка года пройдена: False
Нарушение обнаружено: True
Исходный export не изменился: True


## H+1. Конфликт исходных значений

## H+2. Совпадение опубликованных полей